# 🧙 Fantasy World Predictions — LOTR Universe
## Step 2: Cleaning

---

### 🎯 Goal of this notebook
Take every issue flagged in the EDA notebook and actually fix it, so we end up with a clean
dataframe ready for feature engineering / modeling.

**Decisions carried over from EDA + your call:**
- `gender`, `race`: normalize label variants, **drop rows where still NaN after normalizing**
- `realm`: too sparse to be a strong target — clean it up but keep NaN as `"Unknown"` rather than dropping rows
- `death` → engineer `is_dead` (binary)
- `spouse` → engineer `has_spouse` (binary)
- `hair` → engineer `hair_group` (categorical feature)
- `birth` → engineer `birth_age` (categorical feature: which Age of Middle-earth)
- `height` → drop entirely (89% missing, inconsistent units)

---

## 📦 Section 1: Imports & Load

In [6]:
from pathlib import Path

import pandas as pd
import numpy as np
import re

pd.set_option('display.max_columns', None)

base_dirs = [Path.cwd(), *Path.cwd().parents]
DATA_DIR = next((d / 'data' for d in base_dirs if (d / 'data').exists()), Path.cwd() / 'data')
CSV_PATH = DATA_DIR / 'lotr_characters.csv'

df = pd.read_csv(CSV_PATH, encoding='latin1')
print(f'Starting shape: {df.shape}')

Starting shape: (911, 9)


## 🧑 Section 2: Clean `gender`

Normalize label variants, then drop rows that are still not `Male`/`Female` (includes the true NaNs and anything ambiguous).

In [7]:
df['gender'] = df['gender'].astype(str).str.strip().str.title()
df['gender'] = df['gender'].replace({
    'Males': 'Male',
    'Male ': 'Male',
    'Most Likely Male': 'Male',
    'Nan': np.nan
})

before = len(df)
df = df[df['gender'].isin(['Male', 'Female'])].copy()
print(f'Dropped {before - len(df)} rows with missing/invalid gender. New shape: {df.shape}')
df['gender'].value_counts()

Dropped 143 rows with missing/invalid gender. New shape: (768, 9)


gender
Male      633
Female    135
Name: count, dtype: int64

## 🧝 Section 3: Clean `race`

Merge duplicate labels (Hobbit/Hobbits etc.), group rare races into `Other`, then drop rows still missing race.

In [8]:
race_map = {'Hobbit': 'Hobbits', 'Elf': 'Elves', 'Orc': 'Orcs', 'Maiar': 'Ainur'}
df['race'] = df['race'].astype(str).str.strip().replace(race_map)
df['race'] = df['race'].replace({'Nan': np.nan})

before = len(df)
df = df[df['race'].notna()].copy()
print(f'Dropped {before - len(df)} rows with missing race.')

# Group rare races (fewer than 5 occurrences) as 'Other'
race_counts = df['race'].value_counts()
rare_races = race_counts[race_counts < 5].index
df['race'] = df['race'].apply(lambda x: 'Other' if x in rare_races else x)

print(f'New shape: {df.shape}')
df['race'].value_counts()

Dropped 3 rows with missing race.
New shape: (765, 9)


race
Men           386
Hobbits       142
Elves         103
Other          42
Dwarves        42
Ainur          28
Orcs           10
Half-elven      7
Dragons         5
Name: count, dtype: int64

## 🏰 Section 4: Clean `realm`

Keep the first realm when multiple are listed, group rare realms into `Other`, and fill missing with `"Unknown"` (we're not dropping rows for this one — realm stays a weak, optional target).

In [9]:
def first_realm(val):
    if pd.isna(val):
        return np.nan
    return str(val).split(',')[0].strip()

df['realm'] = df['realm'].apply(first_realm)

realm_counts = df['realm'].value_counts()
top_realms = realm_counts.head(8).index
df['realm'] = df['realm'].apply(
    lambda x: x if (pd.isna(x) or x in top_realms) else 'Other'
)
df['realm'] = df['realm'].fillna('Unknown')

df['realm'].value_counts()

realm
Unknown            570
Other               73
Gondor              33
NÃºmenor            28
Rohan               20
Arthedain           14
Arnor               11
Valinor              6
Shire                5
Lonely Mountain      5
Name: count, dtype: int64

## 💀 Section 5: Engineer `is_dead` from `death`

`0` = alive/immortal, `1` = died. NaN is treated as alive (LOTR wiki convention: no death entry usually means the character is still alive or immortal).

In [10]:
def is_dead(val):
    if pd.isna(val):
        return 0
    if str(val).lower().strip() in ['immortal', 'still alive']:
        return 0
    return 1

df['is_dead'] = df['death'].apply(is_dead)
df['is_dead'].value_counts()

is_dead
1    555
0    210
Name: count, dtype: int64

## 💍 Section 6: Engineer `has_spouse` from `spouse`

`0` = no known spouse, `1` = has a spouse (named or unnamed).

In [11]:
no_spouse_keywords = ['none known', 'unknown, possibly none', 'possibly none',
                       'unknown', 'possibly known', 'nan']

def has_spouse(val):
    if pd.isna(val):
        return 0
    if str(val).lower().strip() in no_spouse_keywords:
        return 0
    return 1

df['has_spouse'] = df['spouse'].apply(has_spouse)
df['has_spouse'].value_counts()

has_spouse
1    450
0    315
Name: count, dtype: int64

## 💇 Section 7: Engineer `hair_group` from `hair`

Bucket free-text hair descriptions into a few broad categories.

In [12]:
def hair_group(val):
    if pd.isna(val):
        return 'Unknown'
    s = str(val).lower()
    if 'dark' in s or 'black' in s:
        return 'Dark'
    if 'golden' in s or 'blond' in s or 'fair' in s:
        return 'Golden'
    if 'light' in s or 'brown' in s:
        return 'Light'
    if 'grey' in s or 'gray' in s or 'white' in s or 'silver' in s:
        return 'Grey/White'
    return 'Other'

df['hair_group'] = df['hair'].apply(hair_group)
df['hair_group'].value_counts()

hair_group
Unknown       594
Dark           69
Golden         44
Light          23
Grey/White     18
Other          17
Name: count, dtype: int64

## 📅 Section 8: Engineer `birth_age` from `birth`

Extract which Age of Middle-earth the character was born in (First/Second/Third/Fourth Age, or Years of the Trees), from very inconsistent free text.

In [13]:
def extract_age(val):
    if pd.isna(val):
        return 'Unknown'
    s = str(val)
    if 'Fourth Age' in s or re.search(r'\bFoA\b', s):
        return 'Fourth Age'
    if 'Third Age' in s or re.search(r'\bTA\b', s):
        return 'Third Age'
    if 'Second Age' in s or re.search(r'\bSA\b', s):
        return 'Second Age'
    if 'First Age' in s or re.search(r'\bFA\b', s):
        return 'First Age'
    if 'Years of the Trees' in s or re.search(r'\bYT\b', s):
        return 'Years of the Trees'
    return 'Unknown'

df['birth_age'] = df['birth'].apply(extract_age)
df['birth_age'].value_counts()

birth_age
Third Age             380
Unknown               144
First Age             101
Second Age             86
Years of the Trees     51
Fourth Age              3
Name: count, dtype: int64

## 🧹 Section 9: Drop raw / unusable columns

`height` is dropped entirely (too sparse, inconsistent units). The raw `death`, `spouse`, `hair`, and `birth` columns are dropped too, since they're now replaced by their engineered versions — keeping both would just duplicate information and confuse the model.

In [14]:
df_clean = df.drop(columns=['height', 'death', 'spouse', 'hair', 'birth'])
df_clean.head()

,gender,name,race,realm,is_dead,has_spouse,hair_group,birth_age
0,Female,Adanel,Men,Unknown,0,1,Unknown,Unknown
1,Male,Boromir,Men,Unknown,1,0,Dark,Third Age
2,Male,Lagduf,Orcs,Unknown,1,0,Unknown,Unknown
3,Male,Tarcil,Men,Arnor,1,1,Unknown,Third Age
4,Male,Fire-drake of Gondolin,Other,Unknown,0,0,Unknown,Unknown


## ✅ Section 10: Final check & save

In [17]:
print('=== Final clean dataset ===')
print(f'Shape: {df_clean.shape}')
print()
print('Dtypes:')
print(df_clean.dtypes)
print()
print('Remaining missing values:')
print(df_clean.isnull().sum())

=== Final clean dataset ===
Shape: (765, 8)

Dtypes:
gender          str
name            str
race            str
realm           str
is_dead       int64
has_spouse    int64
hair_group      str
birth_age       str
dtype: object

Remaining missing values:
gender        0
name          0
race          0
realm         0
is_dead       0
has_spouse    0
hair_group    0
birth_age     0
dtype: int64


In [16]:
output_path = DATA_DIR / 'lotr_characters_clean.csv'
df_clean.to_csv(output_path, index=False)
print(f'✅ Saved to {output_path}')

✅ Saved to c:\Users\hp\Downloads\fantasy-ml-project\fantasy-ml-project\data\lotr_characters_clean.csv


---
## 🔜 What's Next?

Clean data is ready. Next: **Step 3 — `LOTR_ML_Pipeline.ipynb`** — build preprocessing pipelines (encoding, train/test split) and train first models for each candidate target (`gender`, `race`, `realm`, `is_dead`, `has_spouse`).

> **Tell your assistant:** *"Cleaning looks good. Let's move to Step 3 — the ML pipeline."* (or flag anything you want changed first — e.g. different rare-class thresholds, keeping raw columns, etc.)